# AMEX Enterprise Credit Risk Platform
## Notebook 19 — Phase 1, Problem 2: Risk Tier Classification — Business Understanding & Policy
### Problem Statement 2 of 14: Risk Tier Classification (depends on Problem 1's champion PD model)

CRISP-DM stage: **Business Understanding**. First notebook of Problem 2's build (Notebooks 19-25). Problem 2 translates Problem 1's real, calibrated probability-of-default score into discrete risk grades that drive pricing and underwriting policy directly -- per the Master Execution Plan's 14-problem roadmap (Section 8, Phase 1). It depends on Notebook 05's real champion model and cannot run before it.

**What this notebook does, all grounded in Problem 1's real, already-measured results:**

- States Problem 2's business problem, KPI tree, and risk appetite, anchored to Problem 1's real champion model and its real holdout metrics -- not generic boilerplate.
- Defines the **risk tier policy** -- the single source of truth every downstream Problem 2 notebook reads: how many tiers, two real bucketing methods (quantile-based and business-rule/PD-threshold-based) to be computed and compared in Notebook 20, and the KPI targets a valid tier scheme must meet (bad-rate separation, population balance, stability). Every band and threshold is an explicit, editable **ASSUMPTION** -- this dataset carries no ground truth for what a bank's real pricing bands should be.
- Documents stakeholders and their real interest in tier definitions (pricing, underwriting, collections, compliance).

**Deliverables:** `risk_tier_policy.json` (read by every later Problem 2 notebook), `risk_tier_stakeholder_analysis.csv`, and `Risk_Tier_Policy_Charter.docx`.

**Run the single code cell below, once.** Idempotent — every output file is overwritten in place on every re-run.

In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD CONFIG & PROBLEM 1's CHAMPION MODEL (REQUIRED)
# =============================================================================
import os
import sys
import json
import time
import warnings
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Config & Problem 1's Champion Model (Required)")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
CONFIG_PATH = ARTIFACTS_DIR / "project_config.json"
NB05_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_05_summary.json"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"{CONFIG_PATH} not found.\nFix: run 01_business_understanding.ipynb first.")
if not NB05_SUMMARY_PATH.exists():
    raise FileNotFoundError(
        f"{NB05_SUMMARY_PATH} not found.\nProblem 2 (Risk Tier Classification) has a hard dependency on "
        f"Problem 1's champion model -- fix: run 05_model_development.ipynb first (Problem 1, Notebooks 01-18)."
    )

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)
with open(NB05_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB05_SUMMARY = json.load(f)

PILLAR_DIRS = {k: Path(v) for k, v in PROJECT_CONFIG["pillar_dirs"].items()}
RANDOM_SEED = PROJECT_CONFIG["random_seed"]
_resource_limits = PROJECT_CONFIG.get("resource_limits", {})
WARP_THREAD_COUNT = (
    _resource_limits.get("warp_thread_count")
    or PROJECT_CONFIG.get("warp_thread_count")
    or PROJECT_CONFIG["hardware"]["logical_cores_detected"]
)

# --- Self-heal: a project_config.json written before Problem 2 existed will
#     not have these 7 pillar keys yet. Same pattern used by Notebooks 17/18
#     for comprehensive_reporting/repository_packaging -- derive the standard
#     folder, create it, and persist it back so no manual re-run is needed. ---
_REQUIRED_PILLARS = {
    "risk_tier_policy": "02_Problem2_Risk_Tier_Classification/01_Risk_Tier_Policy",
    "risk_tier_modeling": "02_Problem2_Risk_Tier_Classification/02_Risk_Tier_Modeling",
    "risk_tier_validation": "02_Problem2_Risk_Tier_Classification/03_Risk_Tier_Validation",
    "risk_tier_deployment": "02_Problem2_Risk_Tier_Classification/04_Risk_Tier_Deployment",
    "risk_tier_monitoring": "02_Problem2_Risk_Tier_Classification/05_Risk_Tier_Monitoring",
    "risk_tier_reporting": "02_Problem2_Risk_Tier_Classification/06_Risk_Tier_Reporting",
    "risk_tier_packaging": "02_Problem2_Risk_Tier_Classification/07_Risk_Tier_Packaging",
}
_config_healed = False
for _key, _rel_path in _REQUIRED_PILLARS.items():
    if _key not in PILLAR_DIRS:
        PILLAR_DIRS[_key] = PROJECT_ROOT / _rel_path
        PROJECT_CONFIG["pillar_dirs"][_key] = str(PILLAR_DIRS[_key])
        _config_healed = True
        print(f"NOTE: '{_key}' was missing from project_config.json -- added automatically as {PILLAR_DIRS[_key]}")
if _config_healed:
    with open(CONFIG_PATH, "w", encoding="utf-8") as f:
        json.dump(PROJECT_CONFIG, f, indent=2)
    print("\u2705 project_config.json updated in place -- no need to re-run Notebook 01.")

RISK_TIER_POLICY_DIR = PILLAR_DIRS["risk_tier_policy"]
RISK_TIER_POLICY_DIR.mkdir(parents=True, exist_ok=True)

CHAMPION_NAME = NB05_SUMMARY["champion_model"]
CHAMPION_HOLDOUT_AUC = NB05_SUMMARY["champion_metrics"].get("holdout_auc")
CHAMPION_HOLDOUT_AMEX = NB05_SUMMARY["champion_metrics"].get("holdout_amex_metric")

print(f"Problem 1 champion model (real, from Notebook 05): {CHAMPION_NAME}")
print(f"Problem 1 champion holdout AUC (measured)         : {CHAMPION_HOLDOUT_AUC}")
print(f"Problem 1 champion holdout AMEX metric (measured) : {CHAMPION_HOLDOUT_AMEX}")
print(f"Risk tier policy will be written under: {RISK_TIER_POLICY_DIR}")
print("\n\u2705 Section 1 complete.")


# =============================================================================
# SECTION 2: WARP HARDWARE CONFIGURATION, LIBRARY IMPORTS & ADAPTIVE RAM CEILING
# =============================================================================
_section("SECTION 2: WARP Hardware Configuration, Library Imports & Adaptive RAM Ceiling")

os.environ["POLARS_MAX_THREADS"] = str(WARP_THREAD_COUNT)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import logging
logger = logging.getLogger("amex_platform")
logger.setLevel(logging.INFO)
if not logger.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S"))
    logger.addHandler(_handler)

missing = []
try:
    import psutil
except ImportError:
    missing.append("psutil")
try:
    import pandas as pd
except ImportError:
    missing.append("pandas")
try:
    from docx import Document
except ImportError:
    missing.append("python-docx")

if missing:
    raise ImportError(
        "Missing required package(s): " + ", ".join(missing) + "\n"
        "Fix: run this in a terminal, then re-run this cell:\n"
        f"    pip install {' '.join(missing)}"
    )

print("(Reporting only -- this notebook defines policy; it does no data-scale or thread-parallelized work.)")


def _rss_gb() -> float:
    return psutil.Process().memory_info().rss / 1e9


_process_start_rss_gb = _rss_gb()
_live_vm = psutil.virtual_memory()
ADAPTIVE_RAM_FRACTION = _resource_limits.get("ram_fraction_cap", 0.90)
MAX_RAM_BYTES = int(_live_vm.available * ADAPTIVE_RAM_FRACTION)
print(f"Adaptive RAM ceiling (this run) : {MAX_RAM_BYTES / 1e9:.2f} GB")
print(f"\nProcess RSS at Section 2 start: {_process_start_rss_gb:.2f} GB")
print("\n\u2705 Section 2 complete.")


# =============================================================================
# SECTION 3: PROBLEM 2 -- BUSINESS PROBLEM, KPI TREE & RISK APPETITE
# =============================================================================
_section("SECTION 3: Problem 2 -- Business Problem, KPI Tree & Risk Appetite")

# --- Authored problem framing, anchored to Problem 1's real measured metrics
#     above -- this is reference/planning documentation, not computed data,
#     exactly like Notebook 01's own BUSINESS_PROBLEM block and Notebook 15's
#     glossary. It carries no MEASURED/ASSUMPTION label because it asserts no
#     numeric platform fact beyond the real champion metrics it cites. ---
PROBLEM_2_CONTEXT = {
    "problem_name": "Risk Tier Classification",
    "problem_number": 2,
    "phase": "Phase 1 -- Foundation",
    "depends_on": ["Problem 1: Credit Scoring / PD Prediction (Notebooks 01-18)"],
    "problem_statement": (
        f"Problem 1's champion model ({CHAMPION_NAME}, real holdout AUC {CHAMPION_HOLDOUT_AUC}, "
        f"real holdout AMEX metric {CHAMPION_HOLDOUT_AMEX}) outputs a continuous probability of "
        "default per customer. That continuous score is not, by itself, an actionable underwriting "
        "or pricing instrument -- credit policy is written in discrete risk grades (e.g. 'Prime', "
        "'Subprime'), each carrying its own approval rule, pricing, and credit-limit policy. Problem "
        "2 defines a real, validated, monotonic mapping from the continuous PD score to a small set "
        "of risk tiers, and exposes that mapping as a live service."
    ),
    "kpi_tree": [
        "Bad-rate separation: the real, measured default rate of the highest-risk tier must exceed "
        "that of the lowest-risk tier by a wide, statistically significant margin (see KPI target below).",
        "Rank-ordering / monotonicity: real measured bad rate must increase strictly from the lowest "
        "to the highest tier -- a tier scheme that is not monotonic is not usable for pricing.",
        "Population balance: no tier should be so small that it is not commercially actionable, "
        "or so large that it is not differentiated from the whole book.",
        "Stability: tier composition must not swing wildly between two random halves of the same "
        "holdout population -- an unstable scheme cannot support a stable pricing policy.",
    ],
    "risk_appetite_statement": (
        "ASSUMPTION -- this platform's illustrative risk appetite: extend credit across all four "
        "tiers with tier-differentiated pricing and limits, decline or require manual underwriter "
        "override only in the highest-risk tier. A real institution's actual risk appetite is a "
        "board-level policy decision this Kaggle dataset cannot supply."
    ),
}
print(json.dumps(PROBLEM_2_CONTEXT, indent=2))
print("\n\u2705 Section 3 complete.")


# =============================================================================
# SECTION 4: RISK TIER POLICY -- THE SINGLE SOURCE OF TRUTH FOR NOTEBOOKS 20-25
# =============================================================================
_section("SECTION 4: Risk Tier Policy -- The Single Source of Truth for Notebooks 20-25")

# --- Every band, threshold, and KPI target below is an explicit, editable
#     ASSUMPTION -- this dataset has no ground truth for a real bank's pricing
#     policy. The business-rule PD thresholds below intentionally match
#     Notebook 13's illustrative BI bands (Prime <0.05, Near-Prime 0.05-0.15,
#     Subprime 0.15-0.35, High Risk >=0.35) for continuity with Phase 1's
#     existing dashboard -- Problem 2 formally supersedes that illustrative
#     bucketing with a validated policy plus a second, independently-computed
#     quantile method, and Notebook 20 measures and compares both. ---
RISK_TIER_POLICY = {
    "generated_at_utc": None,  # filled in below
    "n_tiers": 4,
    "tier_order": ["Prime", "Near-Prime", "Subprime", "High Risk"],
    "bucketing_methods": {
        "quantile": {
            "description": "ASSUMPTION -- equal-population quartiles of the real holdout PD score "
                            "distribution; guarantees balanced tier sizes by construction.",
            "quantile_cutpoints": [0.25, 0.50, 0.75],
        },
        "business_rule": {
            "description": "ASSUMPTION -- fixed PD thresholds, chosen for continuity with Notebook "
                            "13's illustrative BI bands; population size is whatever the real score "
                            "distribution puts in each band, not guaranteed balanced.",
            "pd_thresholds": [
                {"risk_tier": "Prime", "tier_order": 1, "pd_lower": 0.00, "pd_upper": 0.05},
                {"risk_tier": "Near-Prime", "tier_order": 2, "pd_lower": 0.05, "pd_upper": 0.15},
                {"risk_tier": "Subprime", "tier_order": 3, "pd_lower": 0.15, "pd_upper": 0.35},
                {"risk_tier": "High Risk", "tier_order": 4, "pd_lower": 0.35, "pd_upper": 1.01},
            ],
        },
    },
    "primary_method": "business_rule",  # ASSUMPTION -- which method Notebooks 21-25 treat as canonical
    "kpi_targets": {
        "min_bad_rate_ratio_top_to_bottom_tier": 3.0,   # ASSUMPTION
        "max_tier_population_psi_split_half": 0.10,     # ASSUMPTION -- PSI stability threshold
        "min_tier_population_pct": 5.0,                 # ASSUMPTION -- no tier below 5% of the book
        "require_strict_monotonicity": True,             # ASSUMPTION
    },
    "champion_model_used": CHAMPION_NAME,
    "champion_holdout_auc": CHAMPION_HOLDOUT_AUC,
    "champion_holdout_amex_metric": CHAMPION_HOLDOUT_AMEX,
}
RISK_TIER_POLICY["generated_at_utc"] = datetime.now(timezone.utc).isoformat()

risk_tier_policy_path = RISK_TIER_POLICY_DIR / "risk_tier_policy.json"
with open(risk_tier_policy_path, "w", encoding="utf-8") as f:
    json.dump(RISK_TIER_POLICY, f, indent=2)

print(json.dumps(RISK_TIER_POLICY, indent=2))
print(f"\n\u2705 Saved -> {risk_tier_policy_path}")
print("\n\u2705 Section 4 complete.")


# =============================================================================
# SECTION 5: STAKEHOLDER ANALYSIS
# =============================================================================
_section("SECTION 5: Stakeholder Analysis")

STAKEHOLDERS = [
    {"stakeholder": "Credit Risk / Underwriting", "interest": "Tier definitions directly set approval "
     "and manual-review rules; needs monotonic, statistically defensible bands."},
    {"stakeholder": "Pricing / Product", "interest": "APR and fee schedules are set per tier; needs "
     "stable, well-separated tiers that do not shift materially month to month."},
    {"stakeholder": "Collections", "interest": "Consumes tier as a prioritization signal alongside "
     "delinquency status; needs tier to correlate with real observed loss."},
    {"stakeholder": "Model Risk / Compliance (SR 11-7)", "interest": "Needs the bucketing methodology "
     "documented, validated, and backtested before it can be used for regulatory-adjacent capital or "
     "provisioning inputs (Problem 3, IFRS9/CECL)."},
    {"stakeholder": "Executive / CRO", "interest": "Needs tier-level bad-rate and population reporting "
     "to understand book composition and set risk appetite."},
]
stakeholder_df = pd.DataFrame(STAKEHOLDERS)
stakeholder_path = RISK_TIER_POLICY_DIR / "risk_tier_stakeholder_analysis.csv"
stakeholder_df.to_csv(stakeholder_path, index=False)
print(stakeholder_df.to_string(index=False))
print(f"\u2705 Saved -> {stakeholder_path}")
print("\n\u2705 Section 5 complete.")


# =============================================================================
# SECTION 6: WORD REPORT -- RISK_TIER_POLICY_CHARTER.DOCX
# =============================================================================
_section("SECTION 6: Word Report -- Risk_Tier_Policy_Charter.docx")


def _add_heading(doc, text, level=1):
    return doc.add_heading(text, level=level)


def _add_kv_table(doc, data: dict):
    table = doc.add_table(rows=0, cols=2)
    table.style = "Light Grid Accent 1"
    for k, v in data.items():
        row = table.add_row().cells
        row[0].text = str(k).replace("_", " ").title()
        row[1].text = "" if v is None else str(v)
    return table


def _add_table_from_df(doc, df, max_rows=30):
    table = doc.add_table(rows=1, cols=len(df.columns))
    table.style = "Light Grid Accent 1"
    hdr = table.rows[0].cells
    for i, col in enumerate(df.columns):
        hdr[i].text = str(col).replace("_", " ").title()
    for _, row in df.head(max_rows).iterrows():
        cells_ = table.add_row().cells
        for i, col in enumerate(df.columns):
            cells_[i].text = "" if pd.isna(row[col]) else str(row[col])
    return table


doc = Document()
doc.add_heading("AMEX Enterprise Credit Risk Platform", level=0)
doc.add_paragraph("Phase 1, Problem 2: Risk Tier Classification -- Business Understanding & Policy Charter")
doc.add_paragraph(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

_add_heading(doc, "1. Problem Statement", level=1)
doc.add_paragraph(PROBLEM_2_CONTEXT["problem_statement"])

_add_heading(doc, "2. KPI Tree", level=1)
for _k in PROBLEM_2_CONTEXT["kpi_tree"]:
    doc.add_paragraph(_k, style="List Bullet")

_add_heading(doc, "3. Risk Appetite Statement", level=1)
doc.add_paragraph(PROBLEM_2_CONTEXT["risk_appetite_statement"])

_add_heading(doc, "4. Risk Tier Policy -- Business-Rule Bands (ASSUMPTION)", level=1)
_add_table_from_df(doc, pd.DataFrame(RISK_TIER_POLICY["bucketing_methods"]["business_rule"]["pd_thresholds"]))

_add_heading(doc, "5. Risk Tier Policy -- Quantile Method (ASSUMPTION)", level=1)
doc.add_paragraph(RISK_TIER_POLICY["bucketing_methods"]["quantile"]["description"])
doc.add_paragraph(f"Cutpoints: {RISK_TIER_POLICY['bucketing_methods']['quantile']['quantile_cutpoints']}")

_add_heading(doc, "6. KPI Targets (ASSUMPTION)", level=1)
_add_kv_table(doc, RISK_TIER_POLICY["kpi_targets"])

_add_heading(doc, "7. Stakeholder Analysis", level=1)
_add_table_from_df(doc, stakeholder_df)

report_path = RISK_TIER_POLICY_DIR / "Risk_Tier_Policy_Charter.docx"
doc.save(str(report_path))
print(f"\u2705 Saved -> {report_path}")
print("\n\u2705 Section 6 complete.")


# =============================================================================
# SECTION 7: VERIFICATION
# =============================================================================
_section("SECTION 7: Verification")

_checks_passed = True


def _check(label, condition, detail=""):
    global _checks_passed
    if condition:
        print(f"\u2705 {label}")
    else:
        _checks_passed = False
        print(f"\u274c {label}  {detail}")


_check("Policy defines exactly n_tiers tier names", len(RISK_TIER_POLICY["tier_order"]) == RISK_TIER_POLICY["n_tiers"])
_check("Business-rule PD thresholds are contiguous and cover [0, 1]",
       RISK_TIER_POLICY["bucketing_methods"]["business_rule"]["pd_thresholds"][0]["pd_lower"] == 0.0
       and RISK_TIER_POLICY["bucketing_methods"]["business_rule"]["pd_thresholds"][-1]["pd_upper"] > 1.0)
_check("primary_method is one of the two defined methods",
       RISK_TIER_POLICY["primary_method"] in RISK_TIER_POLICY["bucketing_methods"])
_check("Champion model policy fields match Notebook 05's real summary",
       RISK_TIER_POLICY["champion_model_used"] == CHAMPION_NAME)

_expected_files = [risk_tier_policy_path, stakeholder_path, report_path]
for fp in _expected_files:
    _check(f"{fp.name} exists and is non-empty", fp.exists() and fp.stat().st_size > 0)

if not _checks_passed:
    raise RuntimeError("One or more Notebook 19 verification checks failed. See \u274c lines above.")

print("\nAll Notebook 19 checks passed.")
print("\n\u2705 Section 7 complete.")


# =============================================================================
# SECTION 8: RESOURCE / PERFORMANCE REPORT
# =============================================================================
_section("SECTION 8: Resource / Performance Report")

_final_rss_gb = _rss_gb()
performance_report = {
    "warp_thread_count_configured": WARP_THREAD_COUNT,
    "adaptive_ram_ceiling_gb": round(MAX_RAM_BYTES / 1e9, 2),
    "process_rss_at_start_gb": round(_process_start_rss_gb, 2),
    "process_rss_at_end_gb": round(_final_rss_gb, 2),
}
performance_report_path = ARTIFACTS_DIR / "notebook_19_performance_report.json"
with open(performance_report_path, "w", encoding="utf-8") as f:
    json.dump(performance_report, f, indent=2)
print(f"Process RSS: {_process_start_rss_gb:.2f} GB (start) -> {_final_rss_gb:.2f} GB (end)")
print(f"\u2705 Saved -> {performance_report_path}")
print("\n\u2705 Section 8 complete.")


# =============================================================================
# SECTION 9: WRITE NOTEBOOK 19 SUMMARY ARTIFACT
# =============================================================================
_section("SECTION 9: Write Notebook 19 Summary Artifact")

notebook_19_summary = {
    "notebook": "19_risk_tier_business_understanding", "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "problem_number": 2, "problem_name": "Risk Tier Classification",
    "champion_model": CHAMPION_NAME, "n_tiers": RISK_TIER_POLICY["n_tiers"],
    "primary_method": RISK_TIER_POLICY["primary_method"],
    "output_files": {p.name: str(p) for p in _expected_files + [performance_report_path]},
}
nb19_summary_path = ARTIFACTS_DIR / "notebook_19_summary.json"
with open(nb19_summary_path, "w", encoding="utf-8") as f:
    json.dump(notebook_19_summary, f, indent=2)
print(f"\u2705 Saved -> {nb19_summary_path}")
print("\n\u2705 Section 9 complete.")


# =============================================================================
# SECTION 10: COMPLETION SUMMARY
# =============================================================================
_section("SECTION 10: Notebook 19 Complete -- Handoff to Notebook 20")

print("NOTEBOOK 19: RISK TIER CLASSIFICATION -- BUSINESS UNDERSTANDING & POLICY -- COMPLETE")
print(f"  Champion model (Problem 1, real)  : {CHAMPION_NAME}")
print(f"  Tiers defined                     : {RISK_TIER_POLICY['tier_order']}")
print(f"  Primary bucketing method           : {RISK_TIER_POLICY['primary_method']}")
print(f"  Files produced                    : {len(_expected_files) + 2}")
for _p in _expected_files + [performance_report_path, nb19_summary_path]:
    print(f"    - {_p.name}")
print(f"  Next notebook                     : 20_risk_tier_model_development.ipynb")
print("\n\u2705 Ready to proceed.")
